# CoDaS on All of Us: free biomarker exploration

Run this inside an All of Us Researcher Workbench 2.0 workspace (JupyterLab app), top to bottom.

1. **Setup**: install CoDaS (`aou` branch) and point it at Vertex AI in this workspace's project.
2. **Checks**: the CDR has the Fitbit columns the spec needs; every catalog outcome's concept ids
   exist, sit in the right domain and have the expected names.
3. **Feasibility screen**: participants (and cases) per outcome; counts of 1-20 are masked.
4. **Exploration**: CoDaS runs on the top outcomes and writes masked reports to `.codas_runs/aou/`.

What the model sees: column names, aggregate statistics, and counts with 1-20 masked. Never a row.
The extracted tables stay on this workspace disk (git-ignored). Review every output against the
Data and Statistics Dissemination Policy before anything leaves the Workbench, and do not commit
`.codas_runs/`.

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (REPO / "codas" / "aou").exists(), f"Run this notebook from the codas-adk checkout (got {REPO})."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[aou]"], check=True)
sys.path.insert(0, str(REPO))

In [ ]:
# --- edit if needed -------------------------------------------------------------------------
CDR = os.getenv("CODAS_AOU_CDR") or os.getenv("WORKSPACE_CDR") or ""   # "<cdr-project>.<cdr-dataset>"
REASONING_MODEL = "gemini-3.1-pro-preview"   # or a Claude model id enabled in Vertex Model Garden
FAST_MODEL = "gemini-3-flash-preview"
TOP_K = 3                                    # outcomes to run CoDaS on after the screen
# ---------------------------------------------------------------------------------------------

def sh(*cmd):
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

project = os.getenv("GOOGLE_CLOUD_PROJECT") or sh("gcloud", "config", "get-value", "project")
assert project, "No Google Cloud project found; set GOOGLE_CLOUD_PROJECT to this workspace's project."
assert CDR, ("Set CDR to this workspace's CDR dataset as '<project>.<dataset>'. It appears in the SQL "
             "that the Workbench Dataset Builder generates, or in the workspace's data collection details.")

os.environ.update({
    "CODAS_AOU_MODE": "1",
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "GOOGLE_CLOUD_PROJECT": project,
    "GOOGLE_CLOUD_LOCATION": os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1"),
    "CODAS_AOU_CDR": CDR,
    "CODAS_GEMINI_REASONING_MODEL": REASONING_MODEL,
    "CODAS_GEMINI_FAST_MODEL": FAST_MODEL,
})
for key in ("GOOGLE_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"):
    os.environ.pop(key, None)   # AoU mode refuses consumer API keys

vertex_on = "aiplatform.googleapis.com" in sh("gcloud", "services", "list", "--enabled",
                                               "--filter=name:aiplatform.googleapis.com", "--format=value(name)")
print(f"project: {project}\nCDR: {CDR}\nVertex AI API enabled: {vertex_on}")
assert vertex_on, "Vertex AI (aiplatform.googleapis.com) is not enabled in this workspace project."

from codas.aou.privacy import check_llm_backend
check_llm_backend()
print("model backend: Vertex AI in the workspace project (OK)")

## 2. Checks: CDR schema and outcome catalog

In [ ]:
from google.cloud import bigquery
from IPython.display import Markdown, display

from codas.aou import cohort, feasibility as feas, runner
from codas.aou.cohort import CohortSpec

client = bigquery.Client(project=project)
BASE = CohortSpec.from_json(REPO / "scripts/aou/hba1c_fitbit.example.json")   # wearables, window, demographics
CATALOG = REPO / "scripts/aou/outcomes_catalog.json"

present = cohort._query(client, *cohort.schema_check_sql(BASE, CDR))
missing = cohort.missing_columns(BASE, list(zip(present["table_name"], present["column_name"])))
if missing:
    print("Missing in this CDR:", ", ".join(missing))
    print("Columns that do exist in those tables:")
    for table, cols in present.groupby("table_name")["column_name"]:
        print(f"  {table}: {', '.join(sorted(cols))}")
    raise SystemExit("Edit scripts/aou/hba1c_fitbit.example.json to match, then re-run this cell.")
print("wearable columns: all present")

outcomes = feas.load_catalog(CATALOG)
problems = feas.verify_catalog(outcomes, CDR, client)
display(Markdown("| outcome | concept check |\n|---|---|\n" + "\n".join(
    f"| {o.label} | {problems[o.name] or 'ok'} |" for o in outcomes)))

## 3. Feasibility screen (counts only; 1-20 masked)

In [ ]:
screen = runner.explore(CATALOG, BASE, CDR, client=client, top_k=TOP_K, screen_only=True, log=lambda *_: None)
display(Markdown(screen["table"]))
print("selected for CoDaS:", screen["chosen"] or "none (lower the minimums in runner.explore if needed)")

## 4. Exploration

Runs CoDaS (extraction, profiling, the discovery loop, critic/defender debate, reporting) on each
selected outcome. Expect several minutes per outcome. Only tool names are printed while it runs.

In [ ]:
def progress(agent, tool):
    print(f"  {agent:24s} -> {tool}()")

result = runner.explore(CATALOG, BASE, CDR, client=client, top_k=TOP_K, on_tool_call=progress)
display(Markdown(Path(result["summary_path"]).read_text()))

In [ ]:
for run in result["runs"]:
    display(Markdown(f"---\n## {run['cohort']}\n\n" + Path(run["report_path"]).read_text()))

## Before anything leaves the Workbench

- Outputs are in `.codas_runs/aou/`: per-outcome `*_report.md`, `*_rounds.json`, and `explore_summary.md`.
- Counts of 1-20 are masked by CoDaS, but check every table and figure yourself against the
  Dissemination Policy (including counts that could be derived by subtraction).
- The `*.csv` files are participant-level. Never download, copy, commit or paste them anywhere.